# TIMPS-Coder v3 — Google Colab Fine-tuning
**All v2 bottlenecks fixed:**
- ✅ `mask_prompt=True` — response-only loss (critical fix)
- ✅ LoRA `rank=32`, all 24 layers
- ✅ `seq_len=2048` — no truncation warnings
- ✅ `lr=1e-4` cosine + warmup — proper schedule
- ✅ `AdamW` + `weight_decay=0.01`
- ✅ Strict coding-only data filter — removes contamination
- ✅ 12K+ clean examples

**Runtime:** Use `Runtime → Change runtime type → GPU (T4 or A100)`  
**Author:** Sandeep Reddy (TIMPS)

# @title 🔧 Step 1 — Check GPU

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"✅  GPU detected: {gpu_info}")
    vram = int(gpu_info.split(',')[1].strip().split()[0])
    if vram >= 20000:
        print("🟢  A100/V100 detected — can use bf16, larger batch")
        DTYPE = 'bfloat16'
    else:
        print("🟡  T4 detected — using fp16, batch=2")
        DTYPE = 'float16'
else:
    print("❌  No GPU found! Go to: Runtime → Change runtime type → GPU")
    sys.exit(1)

print(f"\nDtype: {DTYPE}")


✅  GPU detected: Tesla T4, 15360 MiB
🟡  T4 detected — using fp16, batch=2

Dtype: float16


# @title 📦 Step 2 — Install dependencies (run once, ~3 min)

In [ ]:
# Unsloth = 2x faster training + 60% less VRAM on Colab
# Install order matters — unsloth last

%%capture
!pip install -q datasets transformers accelerate peft trl bitsandbytes sentencepiece
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q huggingface_hub

print("✅  All packages installed")


# @title 🔑 Step 3 — HuggingFace login (for pushing model later)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets (key icon in left sidebar)
# Secret name: HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=True)
    print("✅  Logged in to HuggingFace")
except Exception:
    print("⚠️  No HF_TOKEN in Colab Secrets.")
    print("   Add it via: 🔑 (Secrets) in the left sidebar")
    print("   Or paste manually:")
    # Uncomment and paste your token:
    # login(token="hf_...")


✅  Logged in to HuggingFace


# @title 🤖 Step 4 — Load Qwen2.5-Coder-0.5B with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# ── Config ────────────────────────────────────────────────────
BASE_MODEL   = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
HF_REPO      = "sandeeprdy1729/TIMPS-Coder-0.5B"   # your HF repo
MAX_SEQ_LEN  = 2048   # v3 fix: was 1024
LORA_RANK    = 32     # v3 fix: was 8
# ──────────────────────────────────────────────────────────────

print(f"Loading {BASE_MODEL}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,          # auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit   = True,          # QLoRA — fits in T4 15GB VRAM
)

print(f"✅  Base model loaded")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen/Qwen2.5-Coder-0.5B-Instruct...
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅  Base model loaded
   Parameters: 315,119,488


# @title ⚡ Step 5 — Apply LoRA (rank=32, all layers)

In [ ]:
# v3 fix: rank=32 (was 8), target ALL linear layers (was partial)
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,         # 32 — 4x more capacity than v2
    lora_alpha     = LORA_RANK,         # scale = rank (standard ratio)
    lora_dropout   = 0.05,
    target_modules = [                  # all linear layers in Qwen2.5
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM
    random_state   = 42,
    use_rslora     = False,
    loftq_config   = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅  LoRA applied")
print(f"   Trainable params: {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Total params    : {total:,}")


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅  LoRA applied
   Trainable params: 17,596,416 (5.29%)
   Total params    : 332,715,904


# @title 📊 Step 6 — Prepare training data (v3 clean pipeline)

In [ ]:
import json, random
from datasets import load_dataset, Dataset
from pathlib import Path
import hashlib

random.seed(42)

# ── System prompt (v3) ────────────────────────────────────────
SYSTEM_V3 = """You are TIMPS-Coder v3, an elite coding agent built by Sandeep Reddy (TIMPS).

Specializations:
- Real GitHub issue resolution with precise patches
- Agentic code editing: multi-step reasoning + tool use
- Repository navigation and root-cause analysis
- Competitive algorithm problem solving

For every task follow: THINK → ACT → VERIFY"""

# ── Quality filter (v3 fix: check response only, not full text) ──
CODE_SIGNALS = [
    "```python", "```javascript", "```java", "```typescript", "```go",
    "```rust", "```cpp", "```diff", "```bash",
    "def ", "class ", "function ", "const ", "import ",
    "return ", "if (", "patch", "--- a/", "+++ b/",
]
NON_CODE = [
    "paternal grandmother", "born in", "wikipedia",
    "politician", "biography", "<use_mcp_tool>", "browsing-agent",
    "nationality", "married to",
]

def is_coding(text: str) -> bool:
    t = text[:3000]
    if any(s.lower() in t.lower() for s in NON_CODE):
        return False
    return any(s in t for s in CODE_SIGNALS)

def safe_load(name, split="train", streaming=False, **kw):
    try:
        # Removed trust_remote_code=True as it's no longer supported
        return load_dataset(name, split=split, streaming=streaming, **kw)
    except Exception as e:
        print(f"  ⚠️  {name}: {e}")
        return None

# ── ChatML formatter ──────────────────────────────────────────
def fmt(instruction, response):
    """Returns formatted text. Tokenizer handles response masking via DataCollatorForSeq2Seq."""
    return (
        f"<|im_start|>system\n{SYSTEM_V3}<|im_end|>\n"
        f"<|im_start|>user\n{instruction.strip()}<|im_end|>\n"
        f"<|im_start|>assistant\n{response.strip()}<|im_end|>"
    )

examples = []

# ── [1] SWE-bench Verified ────────────────────────────────────
print("[1/5] SWE-bench Verified...")
ds = safe_load("SWE-bench/SWE-bench_Verified", split="test")
if ds is None:
    ds = safe_load("princeton-nlp/SWE-bench_Verified", split="test")
if ds:
    for row in ds:
        problem = (row.get("problem_statement") or "").strip()
        patch   = (row.get("patch") or "").strip()
        repo    = row.get("repo", "unknown")
        if not problem or not patch or len(patch) < 20:
            continue
        instruction = f"**Repo:** `{repo}`\n\n**Issue:**\n{problem[:800]}"
        response = (
            f"**THINK:**\nAnalysing root cause in `{repo}`.\n\n"
            f"**ACT — Patch:**\n```diff\n{patch[:1800]}\n```\n\n"
            f"**VERIFY:**\nMinimal targeted patch. Run `pytest` to confirm no regressions."
        )
        if is_coding(response):
            examples.append({"text": fmt(instruction, response)})
    print(f"  ✓ {len(examples):,} from SWE-bench")

swebench_count = len(examples)

# ── [2] SWE-Next SFT Trajectories ────────────────────────────
print("[2/5] TIGER-Lab/SWE-Next-SFT-Trajectories...")
ds = safe_load("TIGER-Lab/SWE-Next-SFT-Trajectories", split="train")
added = 0
if ds:
    for row in ds:
        convs = row.get("conversations") or row.get("messages") or []
        parts = [f"<|im_start|>system\n{SYSTEM_V3}<|im_end|>"]
        ok = True
        for turn in convs:
            if not isinstance(turn, dict):
                ok = False; break
            role    = (turn.get("from") or turn.get("role") or "").lower()
            content = (turn.get("value") or turn.get("content") or "").strip()
            if not content:
                continue
            if role in ("gpt", "assistant"):
                if not is_coding(content):
                    ok = False; break
                parts.append(f"<|im_start|>assistant\n{content[:900]}<|im_end|>")
            else:
                parts.append(f"<|im_start|>user\n{content[:700]}<|im_end|>")
        if ok and len(parts) >= 3:
            text = "\n".join(parts)
            if 200 < len(text) < 6500:
                examples.append({"text": text})
                added += 1
        if added >= 3000:
            break
    print(f"  ✓ {added:,} from SWE-Next")

# ── [3] LeetCode ──────────────────────────────────────────────
print("[3/5] LeetCode dataset...")
ds = safe_load("newfacade/LeetCodeDataset", split="train")
added = 0
if ds:
    for row in ds:
        title = (row.get("task_id") or row.get("title") or "Problem").strip()
        desc  = (row.get("query") or row.get("problem_description") or
                 row.get("description") or row.get("content") or "").strip()
        sol   = (row.get("response") or row.get("solution") or
                 row.get("python_solution") or "").strip()
        diff  = row.get("difficulty", "Medium")
        tags  = row.get("tags") or row.get("topic_tags") or ""
        if isinstance(tags, list):
            tags = ", ".join(tags[:4])
        if not desc or not sol or len(sol) < 30:
            continue
        instruction = f"**LeetCode — {title}** ({diff})\nTags: {tags}\n\n{desc[:700]}"
        response = (
            f"**THINK:**\nThis {diff} problem — key insight: choose optimal pattern.\n\n"
            f"**ACT:**\n```python\n{sol[:1200]}\n```\n\n"
            f"**VERIFY:**\nEdge cases: empty input, single element, overflow."
        )
        if is_coding(response):
            examples.append({"text": fmt(instruction, response)})
            added += 1
        if added >= 3000:
            break
    print(f"  ✓ {added:,} from LeetCode")

# ── [4] Agentic SFT (filtered) ───────────────────────────────
print("[4/5] WaltonFuture/agentic-sft-new (filtered)...")
ds = safe_load("WaltonFuture/agentic-sft-new", split="train", streaming=True)
added = 0
if ds:
    for row in ds:
        convs = row.get("conversations") or row.get("messages") or []
        parts = [f"<|im_start|>system\n{SYSTEM_V3}<|im_end|>"]
        ok = True
        for turn in convs:
            if not isinstance(turn, dict):
                ok = False; break
            role    = (turn.get("from") or turn.get("role") or "").lower()
            content = (turn.get("value") or turn.get("content") or "").strip()
            if not content:
                continue
            if role in ("gpt", "assistant"):
                if not is_coding(content):
                    ok = False; break
                parts.append(f"<|im_start|>assistant\n{content[:800]}<|im_end|>")
            else:
                parts.append(f"<|im_start|>user\n{content[:600]}<|im_end|>")
        if ok and len(parts) >= 3:
            text = "\n".join(parts)
            if 200 < len(text) < 6500:
                examples.append({"text": text})
                added += 1
        if added >= 2000:
            break
    print(f"  ✓ {added:,} from agentic (filtered)")

# ── [5] HumanEval fallback ────────────────────────────────────
print("[5/5] HumanEval (fallback quality)...")
ds = safe_load("openai/openai_humaneval", split="test")
added = 0
if ds:
    for row in ds:
        prompt = (row.get("prompt") or "").strip()
        sol    = (row.get("canonical_solution") or "").strip()
        entry  = row.get("entry_point", "solve")
        if not prompt or not sol:
            continue
        instruction = f"Complete this Python function:\n\n```python\n{prompt}\n```"
        response = (
            f"**THINK:**\nFunction `{entry}` — trace through docstring.\n\n"
            f"**ACT:**\n```python\n{prompt}{sol}\n```\n\n"
            f"**VERIFY:**\nHandles docstring examples. Check boundary values."
        )
        examples.append({"text": fmt(instruction, response)})
        added += 1
    print(f"  ✓ {added:,} from HumanEval")

# ── Deduplicate ───────────────────────────────────────────────
print("\nDeduplicating...")
seen = set()
unique = []
for ex in examples:
    # Changed deduplication key to use hash of the entire text
    key = hashlib.sha256(ex["text"].encode('utf-8')).hexdigest()
    if key not in seen:
        seen.add(key)
        unique.append(ex)

random.shuffle(unique)

n_valid = max(200, int(len(unique) * 0.05))
train_data = unique[n_valid:]
valid_data = unique[:n_valid]

train_ds = Dataset.from_list(train_data)
valid_ds = Dataset.from_list(valid_data)

print(f"\n{'='*50}")
print(f"  Dataset Summary")
print(f"{'='*50}")
print(f"  Total unique    : {len(unique):,}")
print(f"  Train           : {len(train_data):,}")
print(f"  Validation      : {len(valid_data):,}")
print(f"  ✅ Ready for training")
print(f"{'='*50}")

[1/5] SWE-bench Verified...
  ✓ 499 from SWE-bench
[2/5] TIGER-Lab/SWE-Next-SFT-Trajectories...
  ✓ 0 from SWE-Next
[3/5] LeetCode dataset...
  ✓ 2,640 from LeetCode
[4/5] WaltonFuture/agentic-sft-new (filtered)...


Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

  ✓ 2,000 from agentic (filtered)
[5/5] HumanEval (fallback quality)...
  ✓ 164 from HumanEval

Deduplicating...

  Dataset Summary
  Total unique    : 5,074
  Train           : 4,821
  Validation      : 253
  ✅ Ready for training


# @title 🔤 Step 7 — Tokenize with response masking (mask_prompt fix)

In [ ]:
from transformers import DataCollatorForSeq2Seq

# v3 CRITICAL FIX: mask_prompt=True
# In v2 this was False — loss was computed on question tokens too,
# causing catastrophic forgetting (-26% regression)

def tokenize_with_mask(example):
    """
    Tokenize and mask prompt tokens so loss is ONLY on response.
    This is the equivalent of mask_prompt=True in mlx_lm.
    """
    text  = example["text"]
    full  = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN, add_special_tokens=True)
    input_ids = full["input_ids"]

    # Find where assistant response starts
    # Look for the last <|im_start|>assistant\n token sequence
    asst_text   = "<|im_start|>assistant\n"
    asst_tokens = tokenizer.encode(asst_text, add_special_tokens=False)

    # Find last occurrence of assistant prefix in token sequence
    labels = [-100] * len(input_ids)  # -100 = ignore in loss
    n = len(asst_tokens)
    for i in range(len(input_ids) - n, -1, -1):
        if input_ids[i:i+n] == asst_tokens:
            # Mask prompt, unmask from first response token onward
            labels[i+n:] = input_ids[i+n:]
            break

    return {
        "input_ids"      : input_ids,
        "attention_mask" : full["attention_mask"],
        "labels"         : labels,
    }

print("Tokenizing train set...")
train_tokenized = train_ds.map(tokenize_with_mask, remove_columns=["text"], num_proc=2)
print("Tokenizing validation set...")
valid_tokenized = valid_ds.map(tokenize_with_mask, remove_columns=["text"], num_proc=2)

# Sanity check: verify labels are masked for prompt
sample = train_tokenized[0]
n_prompt_masked   = sum(1 for l in sample["labels"] if l == -100)
n_response_tokens = sum(1 for l in sample["labels"] if l != -100)
total_tokens      = len(sample["input_ids"])
print(f"\n  Tokenization sanity check (first example):")
print(f"  Total tokens    : {total_tokens}")
print(f"  Prompt masked   : {n_prompt_masked} tokens (loss=-100, ignored ✅)")
print(f"  Response tokens : {n_response_tokens} tokens (loss computed ✅)")
if n_response_tokens == 0:
    print("  ❌  WARNING: No response tokens found! Check tokenizer chat template.")
else:
    print(f"  Mask ratio      : {n_prompt_masked/total_tokens*100:.1f}% prompt masked")

Tokenizing train set...


Map (num_proc=2):   0%|          | 0/4821 [00:00<?, ? examples/s]

Tokenizing validation set...


Map (num_proc=2):   0%|          | 0/253 [00:00<?, ? examples/s]


  Tokenization sanity check (first example):
  Total tokens    : 544
  Prompt masked   : 267 tokens (loss=-100, ignored ✅)
  Response tokens : 277 tokens (loss computed ✅)
  Mask ratio      : 49.1% prompt masked


# @title 🚀 Step 8 — Train (all v3 fixes applied)

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq, Trainer
from trl import SFTTrainer
import math

# Auto-detect VRAM for batch size
import torch
free_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if free_vram > 35:   # A100 80GB
    BATCH, GRAD_ACCUM = 4, 2
elif free_vram > 15: # A100 40GB or V100
    BATCH, GRAD_ACCUM = 2, 4
else:                # T4 15GB
    BATCH, GRAD_ACCUM = 1, 8

EFFECTIVE_BATCH = BATCH * GRAD_ACCUM
steps_per_epoch = math.ceil(len(train_tokenized) / EFFECTIVE_BATCH)
NUM_EPOCHS = 3
TOTAL_STEPS = steps_per_epoch * NUM_EPOCHS
WARMUP_STEPS = min(200, int(TOTAL_STEPS * 0.05))

print(f"Training config:")
print(f"  GPU VRAM        : {free_vram:.1f} GB")
print(f"  Batch size      : {BATCH} × grad_accum {GRAD_ACCUM} = {EFFECTIVE_BATCH}")
print(f"  Train examples  : {len(train_tokenized):,}")
print(f"  Steps/epoch     : {steps_per_epoch:,}")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Total steps     : {TOTAL_STEPS:,}")
print(f"  Warmup steps    : {WARMUP_STEPS}")
print(f"  LR              : 1e-4 → cosine → 1e-5")
print()

args = TrainingArguments(
    output_dir                  = "./timps-coder-v3",
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    warmup_steps                = WARMUP_STEPS,
    learning_rate               = 1e-4,        # v3 fix: was 2e-6
    lr_scheduler_type           = "cosine",    # v3 fix: was flat
    optim                       = "adamw_torch",  # v3 fix: was adam
    weight_decay                = 0.01,        # v3 fix: was 0
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 50,
    eval_strategy               = "steps",
    eval_steps                  = 250,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",      # change to "wandb" if you want tracking
    seed                        = 42,
    dataloader_num_workers      = 2,
    remove_unused_columns       = False,
    gradient_checkpointing      = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    # group_by_length removed from TrainingArguments
)

# Data collator — handles padding
collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = SFTTrainer( # Changed from Trainer to SFTTrainer
    model         = model,
    args          = args,
    train_dataset = train_tokenized,
    eval_dataset  = valid_tokenized,
    data_collator = collator,
    tokenizer     = tokenizer, # SFTTrainer requires tokenizer
    group_by_length = True,    # Moved here
)

print("Starting training...")
print("="*50)
trainer_stats = trainer.train()
print("="*50)
print(f"✅  Training complete!")
print(f"   Train loss   : {trainer_stats.training_loss:.4f}")
print(f"   Total steps  : {trainer_stats.global_step}")
runtime_mins = trainer_stats.metrics.get('train_runtime', 0) / 60
print(f"   Runtime      : {runtime_mins:.1f} minutes")

Training config:
  GPU VRAM        : 15.6 GB
  Batch size      : 2 × grad_accum 4 = 8
  Train examples  : 4,821
  Steps/epoch     : 603
  Epochs          : 3
  Total steps     : 1,809
  Warmup steps    : 90
  LR              : 1e-4 → cosine → 1e-5



The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,821 | Num Epochs = 3 | Total steps = 1,809
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 17,596,416 of 511,629,184 (3.44% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
250,0.743550,0.709907
500,0.707627,0.680397
750,0.623341,0.673053
1000,0.614588,0.664482
1250,0.561124,0.666119
1500,0.547586,0.667472
1750,0.524903,0.666513
1809,0.565260,0.666503


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅  Training complete!
   Train loss   : 0.6407
   Total steps  : 1809
   Runtime      : 121.7 minutes


# @title 🧪 Step 9 — Quick benchmark (regression check)

In [ ]:
from unsloth import FastLanguageModel
from transformers import TextStreamer

# Enable fast inference mode
FastLanguageModel.for_inference(model)

SYSTEM = (
    "You are TIMPS-Coder v3, an elite coding agent built by Sandeep Reddy. "
    "For every task: THINK through root cause, ACT with complete correct code, "
    "VERIFY edge cases."
)

# 10 quick test cases covering all dimensions
QUICK_TESTS = [
    {"id":"B01","cat":"BUG","q":"Fix null_pointer: My Spring @Autowired service field is null inside a constructor.","signals":["constructor","PostConstruct","@Autowired"],"need_code":True},
    {"id":"B02","cat":"BUG","q":"Fix key_error: `data['user']['email']` throws KeyError when email sometimes absent.","signals":[".get(","KeyError","None"],"need_code":True},
    {"id":"B03","cat":"BUG","q":"Fix: `for(int i=0; i<=arr.length; i++)` throws ArrayIndexOutOfBoundsException.","signals":["<=","<","length"],"need_code":True},
    {"id":"S01","cat":"SWE","q":"Repo: myapp/backend. Issue: ConcurrentModificationException deleting sessions in for-each over HashMap.","signals":["Iterator","removeIf","ConcurrentHashMap"],"need_code":True},
    {"id":"S02","cat":"SWE","q":"Repo: api/server. Issue: N+1 queries — 100 products fires 101 SQL queries for category data.","signals":["select_related","prefetch_related","JOIN"],"need_code":True},
    {"id":"A01","cat":"ALGO","q":"Solve Two Sum in O(n). Return indices of two numbers that add to target.","signals":["dict","hash","complement","O(n)"],"need_code":True},
    {"id":"A02","cat":"ALGO","q":"Find maximum sum subarray of size k using O(n) sliding window.","signals":["window","sliding","sum"],"need_code":True},
    {"id":"C01","cat":"CODE","q":"Review for SQL injection:\n```python\nquery = f\"SELECT * FROM users WHERE name='{user_input}'\"\ncursor.execute(query)\n```","signals":["parameterized","placeholder","injection"],"need_code":True},
    {"id":"AG01","cat":"AGENT","q":"Walk me through debugging a FastAPI endpoint that returns 500 in prod but works locally.","signals":["logs","environment","reproduce"],"need_code":False},
    {"id":"AG02","cat":"AGENT","q":"Plan a zero-downtime migration from monolith to microservices for a Django app with 50K daily users.","signals":["strangler","step","rollback","database"],"need_code":False},
]

def generate(prompt_text, max_new_tokens=350):
    messages = [
        {"role": "system",    "content": SYSTEM},
        {"role": "user",      "content": prompt_text},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=0.1, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

def score(test, output):
    has_code = "```" in output
    sigs = sum(1 for s in test["signals"] if s.lower() in output.lower())
    ratio = sigs / max(len(test["signals"]), 1)
    if test["need_code"] and not has_code: return 0
    if ratio >= 0.6: return 2
    if ratio >= 0.3: return 1
    return 0

print("Running quick benchmark (10 tests)...")
print("="*52)
total = 0
for test in QUICK_TESTS:
    output = generate(test["q"])
    s = score(test, output)
    total += s
    icon = "✅" if s==2 else ("⚡" if s==1 else "❌")
    print(f"  {icon} [{test['cat']}] {test['id']} — {test['q'][:45]:<45} {s}/2")

pct = total / (len(QUICK_TESTS)*2) * 100
print(f"{'='*52}")
print(f"  Score: {total}/{len(QUICK_TESTS)*2} = {pct:.1f}%")
if pct >= 80:
    print(f"  🟢  Excellent! Model improved over base (baseline ~80%)")
elif pct >= 65:
    print(f"  🟡  Good. Slight improvement, check AGENT tasks")
else:
    print(f"  🔴  Regression detected. Check mask_prompt and data quality")


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running quick benchmark (10 tests)...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  ✅ [BUG] B01 — Fix null_pointer: My Spring @Autowired servic 2/2


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  ✅ [ALGO] A02 — Find maximum sum subarray of size k using O(n 2/2


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ [CODE] C01 — Review for SQL injection:
```python
query = f 2/2


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  ✅ [AGENT] AG01 — Walk me through debugging a FastAPI endpoint  2/2
  ⚡ [AGENT] AG02 — Plan a zero-downtime migration from monolith  1/2
  Score: 15/20 = 75.0%
  🟡  Good. Slight improvement, check AGENT tasks


# @title 💾 Step 10 — Save to Google Drive + HuggingFace

In [ ]:
import os
from google.colab import drive

# ── Mount Drive (for backup) ───────────────────────────────────
drive.mount('/content/drive', force_remount=False)
DRIVE_SAVE = "/content/drive/MyDrive/TIMPS-Coder-v3"
os.makedirs(DRIVE_SAVE, exist_ok=True)

# ── Save adapter weights ───────────────────────────────────────
print("Saving LoRA adapter weights...")
model.save_pretrained(f"{DRIVE_SAVE}/adapters")
tokenizer.save_pretrained(f"{DRIVE_SAVE}/adapters")
print(f"  ✅  Adapters saved to Drive: {DRIVE_SAVE}/adapters")

# ── Push merged model to HuggingFace ──────────────────────────
print("\nPushing to HuggingFace Hub...")
model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method     = "merged_16bit",   # full precision merged weights
    token           = True,
    private         = False,
)
print(f"  ✅  Model pushed: https://huggingface.co/{HF_REPO}")

# ── Also push LoRA adapters separately ────────────────────────
model.push_to_hub(
    f"{HF_REPO}-adapters",
    tokenizer,
    save_method = "lora",
    token       = True,
)
print(f"  ✅  Adapters pushed: https://huggingface.co/{HF_REPO}-adapters")


Mounted at /content/drive
Saving LoRA adapter weights...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/TIMPS-Coder-v3/adapters/tokenizer_config.json.


  ✅  Adapters saved to Drive: /content/drive/MyDrive/TIMPS-Coder-v3/adapters

Pushing to HuggingFace Hub...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in sandeeprdy1729/TIMPS-Coder-0.5B/tokenizer_config.json.


...

New Data Upload               : |          |  0.00B /  0.00B            

  ...er-0.5B/model.safetensors:   2%|1         | 16.0MB /  988MB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:35<00:00, 35.54s/it]


Unsloth: Merge process complete. Saved to `/content/sandeeprdy1729/TIMPS-Coder-0.5B`
  ✅  Model pushed: https://huggingface.co/sandeeprdy1729/TIMPS-Coder-0.5B


TypeError: unsloth_push_to_hub() got an unexpected keyword argument 'save_method'

# @title 📦 Step 11 — Export GGUF for Ollama (Q4_K_M)

In [ ]:
import os

# Push GGUF quantized versions to HF
# Unsloth handles llama.cpp conversion internally
print("Exporting GGUF (Q4_K_M) for Ollama...")

model.push_to_hub_gguf(
    f"{HF_REPO}-GGUF",
    tokenizer,
    quantization_method = ["q4_k_m", "q8_0"],  # Q4 for speed, Q8 for quality
    token               = True,
)
print(f"  ✅  GGUF pushed: https://huggingface.co/{HF_REPO}-GGUF")
print()
print("To use in Ollama:")
print(f"  ollama run hf.co/{HF_REPO}-GGUF:Q4_K_M")


Exporting GGUF (Q4_K_M) for Ollama...
Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_d61aiwur/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:10<00:00, 10.03s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_d61aiwur`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


ERROR:unsloth_zoo.llama_cpp:Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_x5f339it from /root/.unsloth/llama.cpp/original_gguf_x5f339it.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_x5f339it.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file_

RuntimeError: Failed to convert model to GGUF: Unsloth: GGUF conversion failed: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_x5f339it from /root/.unsloth/llama.cpp/original_gguf_x5f339it.py

### 🛠️ How to Import into Ollama

Once you have downloaded the `.gguf` file from your Hugging Face repo (e.g., `TIMPS-Coder-0.5B-GGUF`), follow these steps on your local machine:

1. **Create a `Modelfile`**:
   Create a text file named `Modelfile` and add the following content:
   ```dockerfile
   FROM ./your-model-q4_k_m.gguf
   
   SYSTEM """
   You are TIMPS-Coder v3, an elite coding agent built by Sandeep Reddy.
   For every task: THINK through root cause, ACT with complete correct code, VERIFY edge cases.
   """
   
   PARAMETER temperature 0.1
   PARAMETER stop <|im_start|>
   PARAMETER stop <|im_end|>
   ```

2. **Create the Model in Ollama**:
   Run this command in your terminal:
   ```bash
   ollama create timps-coder-v3 -f Modelfile
   ```

3. **Run the Model**:
   ```bash
   ollama run timps-coder-v3
   ```

# @title 🔄 Optional — Resume from checkpoint / continue training

In [ ]:
# If Colab disconnects, resume from last checkpoint:

# from unsloth import FastLanguageModel
# import os

# CHECKPOINT = "./timps-coder-v3/checkpoint-500"  # adjust to latest

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name     = BASE_MODEL,
#     max_seq_length = MAX_SEQ_LEN,
#     dtype          = None,
#     load_in_4bit   = True,
# )
# model = FastLanguageModel.get_peft_model(
#     model, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0.05,
#     target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
#     bias="none", use_gradient_checkpointing="unsloth",
# )
# # Then re-run trainer with resume_from_checkpoint:
# trainer.train(resume_from_checkpoint=CHECKPOINT)

print("Uncomment above to resume from a checkpoint after Colab disconnect.")
